# Fix3 EDA: Team-scope individual events

Sanity checks after converting 5 match-scope events to team-scope.

In [ ]:
import pandas as pd
labels = pd.read_parquet("../data/processed/labels.parquet")
matches = pd.read_parquet("../data/processed/matches.parquet")

## 1. Row count matches expectation

In [ ]:
print(f"Total rows: {len(labels):,}")
print(f"Expected: ~53,685")
assert len(labels) == 53685, f"Expected 53685, got {len(labels)}"
print("PASS")

## 2. Spot-check: Gayle's 17-six match (2013-04-23, match_id 598027)

In [ ]:
mid = "598027"
for ev in ["any_big_six_hitter", "any_century", "any_fifty"]:
    ev_labels = labels[(labels["match_id"] == mid) & (labels["event_id"] == ev)]
    print(f"{ev}:")
    for _, row in ev_labels.iterrows():
        print(f"  {row['team']}: {row['outcome']}")
# Expected: RCB=1, Pune=0 for all three

## 3. Spot-check: SRH 287 match (2024-04-15, match_id 1426268)

In [ ]:
mid = "1426268"
for ev in ["any_fifty", "any_century"]:
    ev_labels = labels[(labels["match_id"] == mid) & (labels["event_id"] == ev)]
    print(f"{ev}:")
    for _, row in ev_labels.iterrows():
        print(f"  {row['team']}: {row['outcome']}")
# Expected: any_fifty both=1, any_century SRH=1 RCB=0

## 4. Base rate comparison (new team-scope rates)

In [ ]:
for ev in ["any_fifty", "any_century", "any_big_six_hitter", "any_three_wicket_haul", "any_four_wicket_haul"]:
    ev_labels = labels[(labels["event_id"] == ev) & (labels["outcome"].notna())]
    rate = ev_labels["outcome"].mean()
    print(f"{ev:30s}: {rate:.3f} ({rate*100:.1f}%)")

## 5. Independence check

In [ ]:
for ev in ["any_fifty", "any_century", "any_big_six_hitter", "any_three_wicket_haul", "any_four_wicket_haul"]:
    ev_labels = labels[(labels["event_id"] == ev) & (labels["outcome"].notna())]
    merged = ev_labels.merge(matches[["match_id", "team1", "team2"]], on="match_id", how="left")
    t1 = merged[merged["team"] == merged["team1"]][["match_id", "outcome"]].rename(columns={"outcome": "t1_outcome"})
    t2 = merged[merged["team"] == merged["team2"]][["match_id", "outcome"]].rename(columns={"outcome": "t2_outcome"})
    paired = t1.merge(t2, on="match_id", how="inner")
    overall_rate = pd.concat([paired["t1_outcome"], paired["t2_outcome"]]).mean()
    p_t2_given_t1 = paired[paired["t1_outcome"] == 1]["t2_outcome"].mean()
    print(f"{ev:30s}: overall={overall_rate:.3f}, P(t2=1|t1=1)={p_t2_given_t1:.3f}")